# unified_local_llm_server — Demo

One API for ollama, LM Studio, Unsloth, and llama.cpp.

```
server = LLMProviderPool(provider_registry=registry)
llm    = server.load_model("ollama", "gpt-oss:20b")
result = await llm.call(messages=[{"role": "user", "content": "Hi"}])
```

## 1. Setup

In [1]:
from pathlib import Path
from unified_local_llm_server import LLMProviderPool
from unified_local_llm_server.provider_registry import ProviderRegistry

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
registry = ProviderRegistry.load(ROOT / "providers.example.yaml")
server = LLMProviderPool(provider_registry=registry)
print("Providers:", server.get_providers())

Providers: ['llama_cpp', 'lm_studio', 'ollama', 'unsloth']


## 2. Provider Health

In [2]:
statuses = {}
for name in server.get_providers():
    s = await server.check_provider(name)
    statuses[name] = s
    icon = "OK" if s["ok"] else "--"
    print(f"  {icon}  {name:<12}  {s['server_url']}")

  OK  llama_cpp     http://127.0.0.1:9090
  OK  lm_studio     http://127.0.0.1:1234
  OK  ollama        http://127.0.0.1:11434
  OK  unsloth       http://127.0.0.1:8899


## 3. Available Models

In [3]:
for provider in server.get_providers():
    if not statuses[provider]["ok"]:
        continue
    models = server.list_downloaded_models(provider)
    print(f"\n{provider}:")
    if isinstance(models, list):
        for m in models:
            name = m if isinstance(m, str) else m.get("name", m.get("path", m))
            size = f"  {m['total_size_gb']:.1f}GB" if isinstance(m, dict) and 'total_size_gb' in m else ""
            print(f"    {name}{size}")
    else:
        print(f"    {models}")


llama_cpp:
    unsloth_gpt-oss-120b-GGUF_UD-Q8_K_XL_gpt-oss-120b-UD-Q8_K_XL
    DeepSeek-R1-0528-Qwen3-8B-Q4_K_M
    gemma-4-E4B-it-Q4_K_M
    gpt-oss-20b-MXFP4
    Qwen3.6-27B-UD-Q3_K_XL
    Qwen3.6-27B-UD-Q4_K_XL
    gemma-4-26B-A4B-it-UD-Q4_K_XL
    gemma-4-26B-A4B-it-UD-Q4_K_M
    gemma-4-31B-it-UD-Q4_K_XL
    gemma-4-E2B-it-UD-Q4_K_XL
    gpt-oss-20b-UD-Q4_K_XL

lm_studio:
    90f9618340396838ee7ff5b0ba2da27da62953d3
    deepseek/deepseek-r1-0528-qwen3-8b
    google/gemma-4-e4b
    openai/gpt-oss-20b
    text-embedding-nomic-embed-text-v1.5
    82d411acf4a06cfb8d9b073a5211bf410bfc29bf@q3_k_xl
    82d411acf4a06cfb8d9b073a5211bf410bfc29bf@q4_k_xl
    b68961b3c96e42475123a39fe3f8aa149163cf8b@q4_k_m
    b68961b3c96e42475123a39fe3f8aa149163cf8b@q4_k_xl
    8bacec5c8e829a25502cdfe3c3f5b6aabee3218c
    f0c5915f17ad6c66dbeb577fb06ff8925bf8d7ae
    43e80d41a220ac7c83023daacd6a0d1fd8559251@q4_k_xl
    83df0a889143b1dbfc61b591bbc639540fd9ce4c
    d449b42d93e1c2c7bda5312f5c25c8fb91dfa9b4@q4_

## 4. Model Selection

Edit these to match your local models.

In [4]:
MODELS = {
    "ollama":    "gpt-oss:20b",
    "lm_studio": "openai/gpt-oss-20b",
    "unsloth":   "unsloth/gpt-oss-20b-GGUF@UD-Q4_K_XL",
    "llama_cpp": "gpt-oss-20b-MXFP4",
}

## 5. Simple Inference — All Providers

Same prompt, one call per provider.

In [5]:
import time

PROMPT = "Name exactly three benefits of running LLMs locally. Be concise."

for provider, model in MODELS.items():
    if not statuses.get(provider, {}).get("ok"):
        print(f"[{provider}] SKIP")
        continue

    try:
        server.unload_all_models(provider)
        time.sleep(1)
    except Exception:
        pass

    try:
        llm = server.load_model(provider, model)
        t0 = time.perf_counter()
        result = await llm.call(
            messages=[{"role": "user", "content": PROMPT}],
            temperature=0.0,
            options={"num_predict": 120},
        )
        elapsed = round(time.perf_counter() - t0, 1)
        print(f"\n[{provider}]  {elapsed}s")
        print(result)
    except Exception as exc:
        print(f"[{provider}] ERROR: {exc}")


[ollama]  4.7s
- **Privacy & Data Security** – No data leaves the local machine, protecting sensitive information.  
- **Reduced Latency** – Immediate inference without network round‑trips, enabling real‑time applications.  
- **Cost Efficiency** – Eliminates recurring cloud API fees and bandwidth costs.
[lm_studio] ERROR: Chat completions request failed (HTTP 400): '{\n    "error": {\n        "message": "Failed to load model \\"openai/gpt-oss-20b\\". Error: Failed to load model.",\n        "type": "invalid_request_error",\n        "param": "model",\n        "code": null\n    }\n}'

[unsloth]  10.3s
<think>We need to answer: "Name exactly three benefits of running LLMs locally. Be concise." So we need to list exactly three benefits. We must be concise. So maybe bullet points or short sentences. We need to name exactly three benefits. Let's think: 1) Data privacy, 2) Low latency, 3) No dependency on internet or external services. Or 3) No cost of API usage. Or 3) Customization. But we 

## 6. Structured Output

`schema_dict` forces JSON output matching the given shape. A JSON-fix layer retries on parse errors.

In [6]:
import json

# Pick first available provider
active_provider = next((p for p in MODELS if statuses.get(p, {}).get("ok")), None)
if not active_provider:
    print("No provider available")
else:
    llm = server.load_model(active_provider, MODELS[active_provider])

    schema = {
        "model_name":  str,
        "privacy":     bool,
        "speed_rating": int,       # 1-10
        "top_use_cases": [str],
    }

    result = await llm.call(
        messages=[{
            "role": "user",
            "content": "Evaluate running LLMs locally. Fill the schema truthfully.",
        }],
        schema_dict=schema,
        temperature=0.0,
    )

    print(f"Provider: {active_provider}")
    print(json.dumps(result, indent=2))

Provider: ollama
{
  "model_name": "LLaMA 2 7B",
  "privacy": true,
  "speed_rating": 3,
  "top_use_cases": [
    "Text generation",
    "Code generation",
    "Summarization",
    "Translation",
    "Question answering"
  ]
}


## 7. Context Length Control

Pass `context_length` to `load_model`. The provider loads the model with that context window.

In [7]:
provider = active_provider
model    = MODELS[provider]

for ctx in [4096, 32768]:
    try:
        server.unload_all_models(provider)
        time.sleep(1)
    except Exception:
        pass

    llm = server.load_model(provider, model, context_length=ctx)
    result, usage = await llm.call(
        return_usage=True,
        messages=[{"role": "user", "content": "Say OK."}],
        temperature=0.0,
        options={"num_predict": 4},
    )
    prompt_tok = usage.get("prompt_tokens", "?")
    print(f"  ctx={ctx:>6}  prompt_tokens={prompt_tok}  reply={result!r}")

  ctx=  4096  prompt_tokens=91  reply=''
  ctx= 32768  prompt_tokens=91  reply=''


## 8. Batch Calls

`llm.batch()` runs multiple prompts concurrently on the same model handle.

In [8]:
llm = server.load_model(active_provider, MODELS[active_provider])

items = [
    [{"role": "user", "content": "Capital of France?"}],
    [{"role": "user", "content": "Capital of Japan?"}],
    [{"role": "user", "content": "Capital of Brazil?"}],
]

t0 = time.perf_counter()
results = await llm.batch(items, temperature=0.0, options={"num_predict": 8})
elapsed = round(time.perf_counter() - t0, 1)

print(f"Batch of {len(items)} in {elapsed}s:")
for item, res in zip(items, results):
    q = item[0]["content"]
    print(f"  {q:<30} → {res!r}")

Batch of 3 in 2.6s:
  Capital of France?             → 'Paris.'
  Capital of Japan?              → 'Tokyo.'
  Capital of Brazil?             → 'Brasília.'


## 9. Tool Use

The tool pipeline calls Python functions when the model issues tool calls.

In [9]:
def calculator(arguments: dict) -> dict:
    a, b, op = arguments["a"], arguments["b"], arguments.get("op", "+")
    ops = {"+": a + b, "-": a - b, "*": a * b, "/": a / b if b else None}
    return {"result": ops.get(op)}

tools = [{
    "type": "function",
    "function": {
        "name": "calculator",
        "description": "Arithmetic: add, subtract, multiply, divide.",
        "parameters": {
            "type": "object",
            "properties": {
                "a":  {"type": "number"},
                "b":  {"type": "number"},
                "op": {"type": "string", "enum": ["+", "-", "*", "/"]},
            },
            "required": ["a", "b"],
        },
    },
}]

# Only works with providers that support tool calls (not unsloth)
tool_provider = next(
    (p for p in ["ollama", "lm_studio", "llama_cpp"]
     if statuses.get(p, {}).get("ok")), None
)

if not tool_provider:
    print("No tool-capable provider available")
else:
    llm = server.load_model(tool_provider, MODELS[tool_provider])
    result = await llm.call(
        messages=[{"role": "user", "content": "What is 1337 * 42?"}],
        tools=tools,
        tool_registry={"calculator": calculator},
        temperature=0.0,
    )
    print(f"[{tool_provider}] {result}")

[ollama] 56,154


## 10. Multi-Provider Comparison

Same question, all reachable providers, side-by-side.

In [10]:
COMPARE_PROMPT = "In one sentence: what is the main advantage of quantized LLMs?"
comparison = {}

for provider, model in MODELS.items():
    if not statuses.get(provider, {}).get("ok"):
        comparison[provider] = "SKIP"
        continue
    try:
        server.unload_all_models(provider)
        time.sleep(1)
        llm = server.load_model(provider, model)
        t0 = time.perf_counter()
        result = await llm.call(
            messages=[{"role": "user", "content": COMPARE_PROMPT}],
            temperature=0.0,
            options={"num_predict": 60},
        )
        elapsed = round(time.perf_counter() - t0, 1)
        comparison[provider] = (result.strip(), elapsed)
    except Exception as exc:
        comparison[provider] = f"ERROR: {exc}"

print(f"Q: {COMPARE_PROMPT}\n")
for provider, val in comparison.items():
    if isinstance(val, tuple):
        text, elapsed = val
        print(f"[{provider}]  ({elapsed}s)")
        print(f"  {text}\n")
    else:
        print(f"[{provider}]  {val}\n")

Q: In one sentence: what is the main advantage of quantized LLMs?

[ollama]  (4.7s)
  Quantized LLMs dramatically cut memory usage and inference latency—making large models deployable on resource‑constrained devices with only minimal loss in accuracy.

[lm_studio]  ERROR: Chat completions request failed (HTTP 400): '{\n    "error": {\n        "message": "Failed to load model \\"openai/gpt-oss-20b\\". Error: Failed to load model.",\n        "type": "invalid_request_error",\n        "param": "model",\n        "code": null\n    }\n}'

[unsloth]  (9.2s)
  <think>The user asks: "In one sentence: what is the main advantage of quantized LLMs?" They want a one sentence answer. The main advantage: reduced memory usage and faster inference with minimal loss of accuracy. So answer: "They dramatically lower memory footprint and inference latency while

[llama_cpp]  ERROR: Chat completions request failed (HTTP 503): '{"detail":"llama-server is not running — POST /start first"}'

